In [ ]:
import os
import shutil
from google.colab import drive, userdata

# Remove existing directory if it exists
if os.path.exists('/content/mini-gpt'):
    shutil.rmtree('/content/mini-gpt')

# Clone the repository from GitHub
!git clone https://github.com/maariogutierrez/mini-gpt.git
ROOT = '/content/mini-gpt'
os.chdir(ROOT)

# Mount Google Drive for data and outputs
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/My Drive/mini-gpt'

REQUIREMENTS = 'https://raw.githubusercontent.com/maariogutierrez/mini-gpt/main/requirements.txt'
!pip install -r $REQUIREMENTS

wandb_api_key = userdata.get('WANDB')
print("Logging into Weights & Biases (wandb). Follow the prompts.")
import wandb
wandb.login(key=wandb_api_key)

# Store outputs in Google Drive, but keep code in cloned repo
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')
LOGS_DIR = os.path.join(DRIVE_ROOT, 'logs')
DATA_DIR = os.path.join(DRIVE_ROOT, 'data')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')

print("\n--- Setup Summary ---")
print(f"Code repository: {ROOT}")
print(f"Google Drive: {DRIVE_ROOT}")
print(f"Current Working Directory: {os.getcwd()}")

In [2]:

# Preprocess data if it doesn't exist
import sys
sys.path.insert(0, ROOT)

from model.training.preprocess import process_dataset
import os

# Check if train.bin and val.bin exist, if not preprocess
train_bin = os.path.join(DATA_DIR, 'train.bin')
val_bin = os.path.join(DATA_DIR, 'val.bin')

if not os.path.exists(train_bin) or not os.path.exists(val_bin):
    print(f"Preprocessing data to {DATA_DIR}...")
    os.makedirs(DATA_DIR, exist_ok=True)
    process_dataset(output_dir=DATA_DIR, dataset_name="roneneldan/TinyStories")
    print("Preprocessing complete!")
else:
    print(f"Data already exists at {DATA_DIR}")
    print(f"  - train.bin: {os.path.getsize(train_bin) / (1024**3):.2f} GB")
    print(f"  - val.bin: {os.path.getsize(val_bin) / (1024**3):.2f} GB")


Data already exists at /content/drive/My Drive/mini-gpt/data
  - train.bin: 1.52 GB
  - val.bin: 0.17 GB


In [ ]:

# Instantiate model and configure trainer for ~2000 steps
from model.architecture.gpt import GPT, GPTConfig
from model.training.trainer import Trainer
from model.training.dataset import TokenDataset
import torch

# Model configuration - balanced for efficient training
model_config = GPTConfig(
    vocab_size=50257,      # GPT-2 tokenizer vocab size
    block_size=1024,       # Context window
    n_layer=8,             # 8 transformer layers
    n_head=8,              # 8 attention heads
    n_embd=512,            # 512 embedding dimension
    dropout=0.1
)

print("Creating model...")
model = GPT(model_config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

# Load datasets
print("\nLoading datasets...")
train_dataset = TokenDataset(train_bin, model_config.block_size)
val_dataset = TokenDataset(val_bin, model_config.block_size)
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

# Configure trainer for ~2000 steps
# With batch_size=16 and accum_steps=32, we have ~2000 steps to train
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=2,
    accum_steps=4,        # 16 × 32 = 512 effective batch size
    learning_rate=3e-4,    # Moderate learning rate
    weight_decay=0.01,
    warmup_steps=100,      # 100 warmup steps
    max_steps=2000,        # Train for ~2000 steps
    grad_clip=1.0,
    device=device,
    checkpoint_dir=CHECKPOINTS_DIR,
    wandb_project="mini-gpt",
    use_mixed_precision=True,
)

print(f"\nTraining configuration:")
print(f"  Effective batch size: {trainer.effective_batch_size}")
print(f"  Learning rate: {trainer.optimizer.param_groups[0]['lr']:.2e}")
print(f"  Max steps: {trainer.max_steps}")
print(f"  Checkpoint dir: {CHECKPOINTS_DIR}")


Creating model...
GPT Model Information:
  Total parameters: 77,257,809
  Trainable parameters: 77,257,809
  Config:
    - vocab_size: 50257
    - block_size: 1024
    - n_layer: 8
    - n_head: 8
    - n_embd: 512
    - dropout: 0.1
Model parameters: 77,257,809
Device: cuda

Loading datasets...
Train dataset: 407940345 samples
Val dataset: 45325795 samples

Training configuration:
  Effective batch size: 8
  Learning rate: 0.00e+00
  Max steps: 2000
  Checkpoint dir: /content/drive/My Drive/mini-gpt/checkpoints


/content/mini-gpt/model/training/trainer.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if use_mixed_precision else None


: 

In [4]:

# Run training
print("=" * 60)
print("Starting training session: ~2000 steps")
print("=" * 60)

trainer.train()

print("\n" + "=" * 60)
print("Training session complete!")
print("=" * 60)
print(f"Final step: {trainer.global_step}")
print(f"Best validation loss: {trainer.best_val_loss:.4f}")


Starting training session: ~2000 steps


Starting training: 8 effective batch size
(2 micro-batch × 4 accum steps)
Device: cuda



: 

: 